In [1]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import FEMMatrices, PrecomputedGeometricData
from jax_sn.domain.geometric_domain import CrossSectionData, CrossSectionDataSingleGroup
from jax_sn.source import Source
from jax_sn.operators.transport_operator import TransportOperatorBatched
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)
from finite_elements.test_moment_scatter_operator import _setup_all_sb

In [2]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain
from jax_sn.source import Source

In [ ]:
def setup_base_case(dim, degree):
    n_elem       = 5
    n_other_dim  = 10
    other_dims   = [[(2.0, n_other_dim)]]*(dim - 1)
    xs0          = CrossSectionData.constant(0.2, 1, 1)    
    verts, conn, cross_sections, sources = create_cartesian_setup(dim, [[(1.0, n_elem)]] + other_dims, [xs0], [1.0], 0)
    element_type                         = BasixLagrangianSimplex(degree, dim)
    domain                               = Domain.from_mesh_and_cross_sections((verts, conn), element_type.face_template, cross_sections)    
    solution_domain                      = SolutionDomain.from_element_and_domain(element_type, domain)
    # convert source to Source class
    sources = jnp.repeat(sources, repeats = solution_domain.n_basis, axis = -1).reshape(-1, solution_domain.n_basis)

    return solution_domain, sources


solution_domain, sources = setup_base_case(3,2)

i_tn = 50
angle_i = jsn.quadrature_set.create_tn_quadrature_set(4)[50]
toposort = TopologicalSortedAngle.from_domain_angle(solution_domain.domain, angle_i[0], 100)

W1215 14:29:21.726430   13246 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W1215 14:29:21.729629   13196 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


In [12]:
from jax_sn.operators.mass_operator import MassOperator
from jax_sn.operators.mass_operator.mass_operator_base import _mass_kernel_idx_in_precomp_vmap
from jax_sn.operators.transport_operator import TransportOperatorBatched, transport_inverse_sweep_batched, TransportOperatorVmap
import equinox as eqx

In [5]:
transport_operator = TransportOperatorBatched.from_solution_domain_data_and_toposort(solution_domain.solution_domain_data, toposort, batch_size = 10)
mass_operator = MassOperator.from_fem_matrices_and_precomputed_geometric_data(solution_domain.fem_matrices, solution_domain.precomputed_geometric_data)

In [6]:
@jax.jit
def get_transport_operator_inv(sigma, solution_domain_data, toposort, x):
    old_total = solution_domain_data.domain.cross_sections_indexed.cross_section_data.total
    new_total = old_total.at[0,0].set(sigma)
    mod_sol_domain = eqx.tree_at(lambda solution_domain_data : solution_domain_data.domain.cross_sections_indexed.cross_section_data.total, solution_domain_data, new_total)    
    top = TransportOperatorBatched.from_solution_domain_data_and_toposort(mod_sol_domain, toposort, batch_size = 10)
    return top.inv(x,0)

In [43]:
#@jax.jit
def get_transport_operator(sigma, solution_domain_data, toposort, x):    
    old_total = solution_domain_data.domain.cross_sections_indexed.cross_section_data.total
    new_total = old_total.at[0,0].set(sigma)
    mod_sol_domain = eqx.tree_at(lambda solution_domain_data : solution_domain_data.domain.cross_sections_indexed.cross_section_data.total, solution_domain_data, new_total)    
    top = TransportOperatorVmap.from_solution_domain_data_and_toposort(mod_sol_domain, toposort, batch_size = 10)
    return top


top_0 = get_transport_operator(1.0, solution_domain.solution_domain_data, toposort, x= jnp.ones((solution_domain.domain.n_elements, solution_domain.n_basis)))

In [44]:
@jax.jit
def get_transport_operator(sigma, solution_domain_data, toposort, x):    
    old_total = solution_domain_data.domain.cross_sections_indexed.cross_section_data.total
    new_total = old_total.at[0,0].set(sigma)
    mod_sol_domain = eqx.tree_at(lambda solution_domain_data : solution_domain_data.domain.cross_sections_indexed.cross_section_data.total, solution_domain_data, new_total)    
    top = TransportOperatorBatched.from_solution_domain_data_and_toposort(mod_sol_domain, toposort, batch_size = 10)
    return top(x,0)

function_grad = jax.jit(jax.jacfwd(get_transport_operator, argnums=(0)))

In [45]:
x = jnp.ones((solution_domain.domain.n_elements, solution_domain.n_basis))

In [49]:
@jax.jit
def do_stupid_linear_solve(t_operator, energy_group, x):

    def apply_operator(b):
        return t_operator(b, energy_group)

    def t_solve(aa , b):
        return t_operator.inv(b, energy_group)
    
    def tT_solve(T, b):
        return t_operator.T_inv(b, energy_group)
    
    return jax.lax.custom_linear_solve(apply_operator, x, t_solve, tT_solve)


def do_linear_solve(t_op, energy_group, x):
    return t_op.inv(x, energy_group)    


In [53]:
%timeit do_stupid_linear_solve(top_0, 0, x).block_until_ready()

29 ms ± 174 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [52]:
%timeit do_linear_solve(top_0, 0, x).block_until_ready()

29.6 ms ± 331 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
